# TG_NLDAS2_WY2012 fSCA sweep: snow cover fraction by scheme

Compares the snow-covered fraction of the ground (`FSNO`) over Tony Grove, UT, for one
snow season (WY2012: 1 Oct 2011 to 30 Jun 2012), across the `snow_fsca_param` schemes run by
`run_scripts/testing/sweep_fsca.sh`. All variants are clones of the base case
`TG_NLDAS2_WY2012_lastmods` and use its executable, so they differ only in the fSCA scheme.

**Output.** `TG_NLDAS2_WY20XX.sh` sets `hist_nhtfrq = -24` and `hist_mfilt = 365`, so each
variant has one h0 file of daily means, in `$OUTPUT_ROOT/<case>_<variant>/run/`.

For now only CZ26, SL12 and NY07 are loaded (see `VARIANTS` below).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
from dotenv import find_dotenv, load_dotenv

BASE = "TG_NLDAS2_WY2012_lastmods"
VARIANTS = ["CZ26", "SL12", "NY07"]   # add AR25 etc. here once those runs are sorted out
START = "2011-10-01"                  # first day of the history file name

# Paths come from the .env file in the repo root (found by searching up from here)
env_file = find_dotenv(usecwd=True)
if not env_file:
    raise FileNotFoundError("No .env file found in this directory or above it")
load_dotenv(env_file, override=True)
OUTPUT_ROOT = Path(os.environ["OUTPUT_ROOT"])   # model output, one folder per case
# Local copy of the fsurdat file used by the runs (the run script's surfdataFile)
SURFDATA_FILE = Path(os.environ["DATA_ROOT"]) / "surfdata_TG_UT_0125_c261028_merged.nc"

## Load the data

`open_history` opens one variant's h0 file. ELM writes a record at step 0 (the first
half-hour time step), which isn't a daily mean, so it is dropped. Each record's time stamp
is the *end* of its averaging interval (e.g. the 1 Oct mean is stamped 2 Oct 00:00), so the
times are moved to the interval midpoints from `time_bounds`. ELM uses a `noleap` calendar;
converting to `datetime64` makes plotting easier and is safe for these dates (there is no
29 Feb in the noleap calendar, so 29 Feb 2012 is simply missing).

In [ ]:
def open_history(variant):
    case = f"{BASE}_{variant}"
    f = OUTPUT_ROOT / case / "run" / f"{case}.elm.h0.{START}-00000.nc"
    ds = xr.open_dataset(f)
    ds = ds.isel(time=slice(1, None))            # drop the step-0 record
    t0, t1 = ds["time_bounds"].values.T          # cftime objects
    ds = ds.assign_coords(time=t0 + (t1 - t0) / 2)
    return ds.convert_calendar("standard", use_cftime=False)

runs = {v: open_history(v) for v in VARIANTS}
for v, ds in runs.items():
    print(f"{v}: {ds.sizes['time']} days, {ds.time.values[0]} .. {ds.time.values[-1]}, "
          f"grid {ds.sizes['lat']} x {ds.sizes['lon']}")

## Domain-mean fSCA

`FSNO` is the fraction of the ground covered by snow (0 to 1), averaged over the day. The
domain mean is weighted by each grid cell's land area (`area` × `landfrac`), so cells
count by how much land they hold.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for v, ds in runs.items():
    w = ds["area"] * ds["landfrac"]
    ds["FSNO"].weighted(w).mean(("lat", "lon")).plot(ax=ax, label=v)

ax.set(title=f"{BASE}: domain-mean fSCA", xlabel="", ylabel="FSNO (fraction)", ylim=(0, 1.02))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(title="snow_fsca_param")
ax.grid(alpha=0.3)
plt.show()

## fSCA at the highest and lowest grid cells

The domain mean hides the elevation dependence. Here are the grid cells with the highest
and lowest mean elevation, where the schemes may differ the most.

Elevation comes from `TOPO` in the surface data file, not from the history file's `topo`,
which is the atmosphere's surface height from the coupler; DATM leaves it at 0 m. The
surface data file is on dimensions `lsmlat, lsmlon`, the same 12 × 8 grid as the history
files.

In [ ]:
ref = runs[VARIANTS[0]]
surf = xr.open_dataset(SURFDATA_FILE)
# Check that the surfdata grid lines up with the history grid before relabeling it
assert (abs(surf["LATIXY"].isel(lsmlon=0).values - ref.lat.values).max() < 1e-4 and
        abs(surf["LONGXY"].isel(lsmlat=0).values - ref.lon.values).max() < 1e-4)
topo = xr.DataArray(surf["TOPO"].values, coords={"lat": ref.lat, "lon": ref.lon})
cells = {"highest": topo.argmax(("lat", "lon")), "lowest": topo.argmin(("lat", "lon"))}

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, sharey=True)
for ax, (name, idx) in zip(axes, cells.items()):
    for v, ds in runs.items():
        ds["FSNO"].isel(idx).plot(ax=ax, label=v)
    c = topo.isel(idx)
    ax.set(title=f"{name} cell: {float(c):.0f} m "
                 f"({float(c.lat):.2f}N, {float(c.lon) - 360:.2f}E)",
           xlabel="", ylabel="FSNO (fraction)", ylim=(0, 1.02))
    ax.grid(alpha=0.3)
axes[0].legend(title="snow_fsca_param")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()